# 30 - External-Mapped Qwen3-32B QLoRA Fine-tuning

Fine-tunes `Qwen/Qwen3-32B` with external examples mapped to official-law article keys. The locked benchmark remains evaluation-only.

In [ ]:
!python -m pip install -q -U "transformers>=4.51.0" accelerate bitsandbytes peft datasets
!python -m pip uninstall -y torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import sys
import torch

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
sys.path.insert(0, str(DRIVE_ROOT))
sys.path.insert(0, str(DRIVE_ROOT / 'src'))
import os
os.chdir(DRIVE_ROOT)
print('Working directory:', Path.cwd())

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
print('Project:', DRIVE_ROOT)

In [ ]:
from src.finetune_lora import train_lora

model_name = 'Qwen/Qwen3-32B'
train_jsonl = DRIVE_ROOT / 'data/processed/external_mapped_llm_train.jsonl'
val_jsonl = DRIVE_ROOT / 'data/processed/external_mapped_llm_val.jsonl'
output_dir = DRIVE_ROOT / 'models/adapters/qwen3_32b_external_mapped_qlora_v1'

for required in [train_jsonl, val_jsonl]:
    if not required.exists():
        raise FileNotFoundError(required)

print('Train:', train_jsonl)
print('Val:', val_jsonl)
print('Output:', output_dir)

In [ ]:
# If Colab OOMs, reduce max_length to 1536 or use max_train_samples for a shorter diagnostic run.
run_config = train_lora(
    train_jsonl=train_jsonl,
    val_jsonl=val_jsonl,
    output_dir=output_dir,
    model_name=model_name,
    max_length=2048,
    max_train_samples=None,
    max_val_samples=None,
    num_train_epochs=1.0,
    learning_rate=1e-4,
    gradient_accumulation_steps=16,
    use_4bit=True,
)
print(json.dumps(run_config, ensure_ascii=False, indent=2))